# VHAGAR T2 - Prithvi CONUS->Europe transfer (train CONUS, predict Europe)

One fresh run: fine-tune Prithvi on the CONUS burn-balanced chips, then predict the
European (EMS) chips with that same checkpoint. No restart; subprocess training.

**Upload two zips** (Files panel): `t2_prithvi_chips.zip` (CONUS, burn-balanced) and
`emsr_chips.zip` (European inference chips). Set Runtime -> GPU. Then Run all.


## 1. GPU + install


In [ ]:
!nvidia-smi -L
!pip install -q terratorch "torchgeo==0.7.1" "numpy==2.2.6"
print('installed')


## 2. Parameters


In [ ]:
CONUS_ZIP  = '/content/t2_prithvi_chips.zip'
EU_ZIP     = '/content/emsr_chips.zip'
LOSS       = 'dice'
MAX_EPOCHS = 60


## 3. Unzip both datasets


In [ ]:
!rm -rf /content/t2_prithvi_chips /content/t2_prithvi_emsr_chips
!unzip -q -o "$CONUS_ZIP" -d /content/t2_prithvi_chips
!unzip -q -o "$EU_ZIP"    -d /content/t2_prithvi_emsr_chips
!echo CONUS train: $(wc -l < /content/t2_prithvi_chips/splits/train.txt) '| EU chips:' $(ls /content/t2_prithvi_emsr_chips/data/*_merged.tif | wc -l)


## 4. Write the CONUS config


In [ ]:
cfg = r'''seed_everything: 2
trainer:
  max_epochs: MAX_EPOCHS_PLACEHOLDER
  log_every_n_steps: 5
  callbacks:
    - class_path: EarlyStopping
      init_args: {monitor: val/loss, patience: 12}
    - class_path: ModelCheckpoint
      init_args: {monitor: val/loss, save_top_k: 1, filename: best}
  precision: bf16-mixed
model:
  class_path: terratorch.tasks.SemanticSegmentationTask
  init_args:
    model_factory: EncoderDecoderFactory
    model_args:
      backbone: prithvi_eo_v2_300
      backbone_pretrained: true
      backbone_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
      necks:
        - {name: SelectIndices, indices: [5, 11, 17, 23]}
        - {name: ReshapeTokensToImage}
        - {name: LearnedInterpolateToPyramidal}
      decoder: UNetDecoder
      decoder_channels: [512, 256, 128, 64]
      num_classes: 2
    loss: LOSS_PLACEHOLDER
    ignore_index: -1
    freeze_backbone: false
    class_names: [Not burned, Burn scar]
optimizer:
  class_path: torch.optim.AdamW
  init_args: {lr: 1.e-4}
lr_scheduler:
  class_path: ReduceLROnPlateau
  init_args: {monitor: val/loss, factor: 0.5, patience: 4}
data:
  class_path: GenericNonGeoSegmentationDataModule
  init_args:
    batch_size: 8
    num_workers: 2
    dataset_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    output_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    rgb_indices: [2, 1, 0]
    train_data_root: /content/t2_prithvi_chips/data
    val_data_root: /content/t2_prithvi_chips/data
    test_data_root: /content/t2_prithvi_chips/data
    train_split: /content/t2_prithvi_chips/splits/train.txt
    val_split: /content/t2_prithvi_chips/splits/val.txt
    test_split: /content/t2_prithvi_chips/splits/test.txt
    img_grep: "*_merged.tif"
    label_grep: "*.mask.tif"
    means: [0.049771, 0.068685, 0.076481, 0.214572, 0.204851, 0.146201]
    stds:  [0.031063, 0.036317, 0.049595, 0.085017, 0.094316, 0.082628]
    num_classes: 2
    train_transform:
      - {class_path: albumentations.D4}
      - {class_path: ToTensorV2}
    test_transform:
      - {class_path: ToTensorV2}
    no_data_replace: 0
    no_label_replace: -1
'''.replace('MAX_EPOCHS_PLACEHOLDER', str(MAX_EPOCHS)).replace('LOSS_PLACEHOLDER', LOSS)
open('/content/prithvi_conus.yaml','w').write(cfg)
print('config written: loss =', LOSS, '| max_epochs =', MAX_EPOCHS)


## 5. Fine-tune on CONUS (GPU)


In [ ]:
!terratorch fit -c /content/prithvi_conus.yaml


## 6. (optional) CONUS test metric


In [ ]:
import glob
ckpt = (sorted(glob.glob('/content/**/best*.ckpt', recursive=True)))[-1]; print(ckpt)
!terratorch test -c /content/prithvi_conus.yaml --ckpt_path "$ckpt"


## 7. Predict the European chips with the CONUS checkpoint
The predict runs as a **subprocess** (writes masks + zip). The download runs **in-kernel**
on the next line - `files.download` cannot run inside a subprocess (no kernel there).


In [ ]:
open('/content/predict_eu.py','w').write(r'''import glob, os, numpy as np, rasterio, torch, shutil
from terratorch.tasks import SemanticSegmentationTask
ck = sorted(glob.glob("/content/**/best*.ckpt", recursive=True))[-1]; print("CONUS ckpt:", ck)
task = SemanticSegmentationTask.load_from_checkpoint(ck, map_location="cuda").eval()
means = np.array([0.049771,0.068685,0.076481,0.214572,0.204851,0.146201],"float32")[:,None,None]
stds  = np.array([0.031063,0.036317,0.049595,0.085017,0.094316,0.082628],"float32")[:,None,None]
stems = [l.strip() for l in open("/content/t2_prithvi_emsr_chips/splits/all.txt") if l.strip()]
os.makedirs("/content/eu_preds", exist_ok=True)
def to_logits(o):
    o = getattr(o, "output", o); return torch.as_tensor(o[0] if isinstance(o,(list,tuple)) else o)
for stem in stems:
    with rasterio.open(f"/content/t2_prithvi_emsr_chips/data/{stem}_merged.tif") as s: img = s.read().astype("float32")
    x = torch.from_numpy(((img - means)/stds)[None]).to("cuda")
    with torch.no_grad(): pred = to_logits(task(x)).float().argmax(1)[0].cpu().numpy().astype("int16")
    with rasterio.open(f"/content/eu_preds/{stem}.tif","w",driver="GTiff",height=pred.shape[0],width=pred.shape[1],count=1,dtype="int16") as d: d.write(pred[None])
print("wrote", len(stems), "European masks")
shutil.make_archive("/content/prithvi_eu_preds", "zip", "/content/eu_preds")
''')
!python /content/predict_eu.py


In [ ]:
# in-kernel download (must NOT be inside the subprocess above)
from google.colab import files; files.download('/content/prithvi_eu_preds.zip')


## 8. Back on your machine: the transfer verdict
```
Expand-Archive "$env:USERPROFILE\Downloads\prithvi_eu_preds.zip" -DestinationPath data\prithvi_eu_preds -Force
vhagar t2-prithvi-transfer --pred-dir data\prithvi_eu_preds \
    --chips-manifest data\t2_prithvi_emsr_chips\_chips.json
```
CONUS-trained Prithvi vs the CONUS-tuned NBR threshold, on the European fires.
